# Tiền xử lí dữ liệu
Bộ dữ liệu **Dry_Bean_Dataset.csv**

## Import các thư viện cần thiết

In [1]:
import pandas as pd
import numpy as np
import matplotlib as plt
import seaborn as sns
import pickle
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import PowerTransformer, RobustScaler, LabelEncoder
from sklearn.utils.class_weight import compute_class_weight

## Đọc dữ liệu từ file `Dry_Bean_Dataset.csv`

In [2]:
DATA_PATH = "../../data/classification"

df = pd.read_excel(DATA_PATH + "/Dry_Bean_Dataset.xlsx")
df.head()

,Area,Perimeter,MajorAxisLength,MinorAxisLength,AspectRation,Eccentricity,ConvexArea,EquivDiameter,Extent,Solidity,roundness,Compactness,ShapeFactor1,ShapeFactor2,ShapeFactor3,ShapeFactor4,Class
0,28395,610.291,208.178117,173.888747,1.197191,0.549812,28715,190.141097,0.763923,0.988856,0.958027,0.913358,0.007332,0.003147,0.834222,0.998724,SEKER
1,28734,638.018,200.524796,182.734419,1.097356,0.411785,29172,191.272750,0.783968,0.984986,0.887034,0.953861,0.006979,0.003564,0.909851,0.998430,SEKER
2,29380,624.110,212.826130,175.931143,1.209713,0.562727,29690,193.410904,0.778113,0.989559,0.947849,0.908774,0.007244,0.003048,0.825871,0.999066,SEKER
3,30008,645.884,210.557999,182.516516,1.153638,0.498616,30724,195.467062,0.782681,0.976696,0.903936,0.928329,0.007017,0.003215,0.861794,0.994199,SEKER
4,30140,620.134,201.847882,190.279279,1.060798,0.333680,30417,195.896503,0.773098,0.990893,0.984877,0.970516,0.006697,0.003665,0.941900,0.999166,SEKER


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13611 entries, 0 to 13610
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Area             13611 non-null  int64  
 1   Perimeter        13611 non-null  float64
 2   MajorAxisLength  13611 non-null  float64
 3   MinorAxisLength  13611 non-null  float64
 4   AspectRation     13611 non-null  float64
 5   Eccentricity     13611 non-null  float64
 6   ConvexArea       13611 non-null  int64  
 7   EquivDiameter    13611 non-null  float64
 8   Extent           13611 non-null  float64
 9   Solidity         13611 non-null  float64
 10  roundness        13611 non-null  float64
 11  Compactness      13611 non-null  float64
 12  ShapeFactor1     13611 non-null  float64
 13  ShapeFactor2     13611 non-null  float64
 14  ShapeFactor3     13611 non-null  float64
 15  ShapeFactor4     13611 non-null  float64
 16  Class            13611 non-null  object 
dtypes: float64(1

In [4]:
df['Class'].value_counts()

Class
DERMASON    3546
SIRA        2636
SEKER       2027
HOROZ       1928
CALI        1630
BARBUNYA    1322
BOMBAY       522
Name: count, dtype: int64

## Pipeline tiền xử lí chi tiết (model-agnostic)
Mục tiêu: tạo ra dữ liệu sạch, không rò rỉ thông tin (data leakage), sẵn sàng cho mọi mô hình.

Các bước chính:
1. Chuẩn hoá tên cột và schema.
2. Làm sạch nhãn `Class`, xử lí trùng lặp.
3. Chia train/validation/test theo stratified.
4. Fit bộ xử lí chỉ trên train: IQR capping (outlier), Power Transform, Robust Scaling.
5. Sinh 2 phiên bản đặc trưng:
   - `raw` (đã làm sạch + capping): phù hợp tree-based.
   - `scaled` (thêm power + robust scaler): phù hợp SVM/kNN/Logistic/NN.
6. Tạo class weights để xử lí mất cân bằng lớp ở bước huấn luyện.

In [5]:
df_work = df.copy()

expected_columns = [
    "Area", "Perimeter", "MajorAxisLength", "MinorAxisLength", "AspectRation", "Eccentricity",
    "ConvexArea", "EquivDiameter", "Extent", "Solidity", "roundness", "Compactness",
    "ShapeFactor1", "ShapeFactor2", "ShapeFactor3", "ShapeFactor4", "Class"
]

# Clean Class labels and drop duplicates
before_rows = len(df_work)
df_work["Class"] = df_work["Class"].astype(str).str.strip().str.upper()
df_work = df_work.dropna(subset=["Class"])
df_work = df_work.drop_duplicates().reset_index(drop=True)

# cast feature columns to numeric, drop rows with parsing errors (if any)
feature_cols = [c for c in expected_columns if c != "Class"]
df_work[feature_cols] = df_work[feature_cols].apply(pd.to_numeric, errors="coerce")
num_na_after_coerce = int(df_work[feature_cols].isna().sum().sum())
if num_na_after_coerce > 0:
    df_work = df_work.dropna(subset=feature_cols).reset_index(drop=True)

print("Rows before cleaning:", before_rows)
print("Rows after cleaning :", len(df_work))
print("Classes:", df_work["Class"].value_counts().to_dict())

Rows before cleaning: 13611
Rows after cleaning : 13543
Classes: {'DERMASON': 3546, 'SIRA': 2636, 'SEKER': 2027, 'HOROZ': 1860, 'CALI': 1630, 'BARBUNYA': 1322, 'BOMBAY': 522}


In [6]:
# Split dataset: train/val/test
X = df_work[feature_cols].copy()
y = df_work["Class"].copy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y,
    test_size=0.30,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42
)

print("Train shape:", X_train.shape, "| class dist:", y_train.value_counts(normalize=True).round(4).to_dict())
print("Val shape  :", X_val.shape, "| class dist:", y_val.value_counts(normalize=True).round(4).to_dict())
print("Test shape :", X_test.shape, "| class dist:", y_test.value_counts(normalize=True).round(4).to_dict())

Train shape: (9480, 16) | class dist: {'DERMASON': 0.2618, 'SIRA': 0.1946, 'SEKER': 0.1497, 'HOROZ': 0.1373, 'CALI': 0.1204, 'BARBUNYA': 0.0976, 'BOMBAY': 0.0386}
Val shape  : (2031, 16) | class dist: {'DERMASON': 0.2619, 'SIRA': 0.1945, 'SEKER': 0.1497, 'HOROZ': 0.1374, 'CALI': 0.1201, 'BARBUNYA': 0.098, 'BOMBAY': 0.0384}
Test shape : (2032, 16) | class dist: {'DERMASON': 0.2618, 'SIRA': 0.1949, 'SEKER': 0.1496, 'HOROZ': 0.1373, 'CALI': 0.1206, 'BARBUNYA': 0.0974, 'BOMBAY': 0.0384}


In [7]:
# Fit preprocessors 

def fit_iqr_caps(df_num: pd.DataFrame, k: float = 1.5):
    q1 = df_num.quantile(0.25)
    q3 = df_num.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - k * iqr
    upper = q3 + k * iqr
    return lower, upper


def apply_iqr_caps(df_num: pd.DataFrame, lower: pd.Series, upper: pd.Series):
    return df_num.clip(lower=lower, upper=upper, axis=1)

# IQR capping
iqr_lower, iqr_upper = fit_iqr_caps(X_train, k=1.5)
X_train_raw = apply_iqr_caps(X_train, iqr_lower, iqr_upper)
X_val_raw = apply_iqr_caps(X_val, iqr_lower, iqr_upper)
X_test_raw = apply_iqr_caps(X_test, iqr_lower, iqr_upper)

# Power transform to reduce skewness 
power_tf = PowerTransformer(method="yeo-johnson", standardize=False)
X_train_pw = pd.DataFrame(power_tf.fit_transform(X_train_raw), columns=feature_cols, index=X_train_raw.index)
X_val_pw = pd.DataFrame(power_tf.transform(X_val_raw), columns=feature_cols, index=X_val_raw.index)
X_test_pw = pd.DataFrame(power_tf.transform(X_test_raw), columns=feature_cols, index=X_test_raw.index)

# Robust scaling
robust_scaler = RobustScaler()
X_train_scaled = pd.DataFrame(robust_scaler.fit_transform(X_train_pw), columns=feature_cols, index=X_train_pw.index)
X_val_scaled = pd.DataFrame(robust_scaler.transform(X_val_pw), columns=feature_cols, index=X_val_pw.index)
X_test_scaled = pd.DataFrame(robust_scaler.transform(X_test_pw), columns=feature_cols, index=X_test_pw.index)

print("Preprocessing complete.")
print("raw train shape   :", X_train_raw.shape)
print("scaled train shape:", X_train_scaled.shape)

Preprocessing complete.
raw train shape   : (9480, 16)
scaled train shape: (9480, 16)


/home/lesliu/miniforge3/envs/intro2ml/lib/python3.10/site-packages/numpy/core/_methods.py:176: RuntimeWarning: overflow encountered in multiply
  x = um.multiply(x, x, out=x)


In [8]:
# 5) Encoding labels + class weights
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_val_enc = label_encoder.transform(y_val)
y_test_enc = label_encoder.transform(y_test)

classes = label_encoder.classes_
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_map = {cls: float(w) for cls, w in zip(classes, weights)}

print("Label mapping:", {i: cls for i, cls in enumerate(classes)})
print("Class weights:", class_weight_map)

Label mapping: {0: 'BARBUNYA', 1: 'BOMBAY', 2: 'CALI', 3: 'DERMASON', 4: 'HOROZ', 5: 'SEKER', 6: 'SIRA'}
Class weights: {'BARBUNYA': 1.4640926640926641, 'BOMBAY': 3.700234192037471, 'CALI': 1.186928759233755, 'DERMASON': 0.5456429147001266, 'HOROZ': 1.0401579986833442, 'SEKER': 0.9543944427665358, 'SIRA': 0.7340301974448316}


In [9]:
# Summary of preprocessed artifacts:
# - Tree-based: *_raw
# - Distance/Linear/NN: *_scaled
preprocessed = {
    "X_train_raw": X_train_raw,
    "X_val_raw": X_val_raw,
    "X_test_raw": X_test_raw,
    "X_train_scaled": X_train_scaled,
    "X_val_scaled": X_val_scaled,
    "X_test_scaled": X_test_scaled,
    "y_train": y_train,
    "y_val": y_val,
    "y_test": y_test,
    "y_train_enc": y_train_enc,
    "y_val_enc": y_val_enc,
    "y_test_enc": y_test_enc,
    "feature_cols": feature_cols,
    "label_classes": classes,
    "class_weight_map": class_weight_map,
    "iqr_lower": iqr_lower,
    "iqr_upper": iqr_upper,
    "power_transformer": power_tf,
    "robust_scaler": robust_scaler,
    "label_encoder": label_encoder,
}

print("Artifacts ready for training step.") 
print("Keys:", list(preprocessed.keys()))

Artifacts ready for training step.
Keys: ['X_train_raw', 'X_val_raw', 'X_test_raw', 'X_train_scaled', 'X_val_scaled', 'X_test_scaled', 'y_train', 'y_val', 'y_test', 'y_train_enc', 'y_val_enc', 'y_test_enc', 'feature_cols', 'label_classes', 'class_weight_map', 'iqr_lower', 'iqr_upper', 'power_transformer', 'robust_scaler', 'label_encoder']


In [10]:
# Save preprocessed datasets
DATA_PATH = Path(str(DATA_PATH)).resolve()

if DATA_PATH is not None:
    project_root = DATA_PATH.parent.parent
    out_dir = DATA_PATH / "processed"
    out_dir.mkdir(parents=True, exist_ok=True)

    # Save train/val/test datasets (raw & scaled)
    pd.concat([X_train_raw, y_train.rename("Class")], axis=1).to_csv(out_dir / "train_raw.csv", index=False)
    pd.concat([X_val_raw, y_val.rename("Class")], axis=1).to_csv(out_dir / "val_raw.csv", index=False)
    pd.concat([X_test_raw, y_test.rename("Class")], axis=1).to_csv(out_dir / "test_raw.csv", index=False)

    pd.concat([X_train_scaled, y_train.rename("Class")], axis=1).to_csv(out_dir / "train_scaled.csv", index=False)
    pd.concat([X_val_scaled, y_val.rename("Class")], axis=1).to_csv(out_dir / "val_scaled.csv", index=False)
    pd.concat([X_test_scaled, y_test.rename("Class")], axis=1).to_csv(out_dir / "test_scaled.csv", index=False)
    
    with open(out_dir / "preprocessed.pkl", "wb") as f:
        pickle.dump(preprocessed, f)

    print("Saved preprocessed dict to:", out_dir / "preprocessed.pkl")

    print("Saved preprocessed datasets to:", out_dir)
else:
    print("Could not determine the path to the original CSV file.")

Saved preprocessed dict to: /home/lesliu/Documents/school/25_26_Semester_2/intro2ml/lab1/data/classification/processed/preprocessed.pkl
Saved preprocessed datasets to: /home/lesliu/Documents/school/25_26_Semester_2/intro2ml/lab1/data/classification/processed
